# LABORATORIO 2: Modelos y métricas
INTEGRANTES: Benjamín Aballay / Felipe Martínez \
Asignatura: Computación paralela y distribuida(INFB8090). \
Docente: Michael Miranda \
Sección: 412 \
Fecha: 23 de Abril \
Institución: Universidad Tecnológica Metropolitana

## Información del entorno de ejecución

In [7]:
import platform, os, multiprocessing, sys

print('=== Configuración del sistema ===')
print(f'Sistema operativo : {platform.system()} {platform.release()}')
print(f'Procesador        : {platform.processor()}')
print(f'Núcleos lógicos   : {multiprocessing.cpu_count()}')
print(f'Python            : {sys.version}')

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024**3)
    print(f'RAM aproximada    : {ram_gb:.1f} GB')
except ImportError:
    print('RAM aproximada    : instala psutil para verla (pip install psutil)')

=== Configuración del sistema ===
Sistema operativo : Windows 11
Procesador        : AMD64 Family 25 Model 33 Stepping 2, AuthenticAMD
Núcleos lógicos   : 12
Python            : 3.13.2 | packaged by Anaconda, Inc. | (main, Feb  6 2025, 18:49:14) [MSC v.1929 64 bit (AMD64)]
RAM aproximada    : 31.9 GB


---
## Ejercicio 1 — Línea base y comparación vectorizada
**Propósito:** Establecer una línea base secuencial y compararla con operaciones vectorizadas (NumPy),
para distinguir entre *optimización algorítmica* y *paralelismo* real.

### 1a) Versión secuencial (bucle Python puro)

In [8]:
import time
import math
import numpy as np
import sympy as sp
import pandas as pd
from IPython.display import display, HTML

# -------------------------------------------------------
# 1. Renderizado de la fórmula matemática con SymPy
# -------------------------------------------------------
display(HTML("<h3 style='color: #2E86C1;'>Fórmula Utilizada:</h3>"))
x, n_sym = sp.symbols('x n')
formula = sp.Sum(sp.sqrt(x) + sp.log(1 + x), (x, 1, n_sym))
display(formula)

# -------------------------------------------------------
# Función 1a: implementación secuencial con bucle Python
# f(x) = sqrt(x) + log(1+x)  para x en {1..n}
# -------------------------------------------------------
def suma_transformacion_secuencial(n: int) -> float:
    """Calcula sum_{x=1}^{n} [ sqrt(x) + log(1+x) ] con un bucle Python."""
    acumulador = 0.0
    for x in range(1, n + 1):
        acumulador += math.sqrt(x) + math.log(1 + x)
    return acumulador

def medir_tiempo(funcion, *args, repeticiones: int = 3) -> float:
    """Ejecuta `funcion(*args)` varias veces y devuelve el tiempo mínimo (segundos)."""
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion(*args)
        tiempos.append(time.perf_counter() - inicio)
    return min(tiempos)

tamanios = [1000000, 3000000, 6000000, 7000000, 8000000, 10000000]
resultados_1a = []

for n in tamanios:
    t_sec = medir_tiempo(suma_transformacion_secuencial, n)
    resultados_1a.append({'n': n, 't_sec': t_sec})

# -------------------------------------------------------
# 2. Visualización
# -------------------------------------------------------
df_1a = pd.DataFrame(resultados_1a)
df_1a.columns = ["Tamaño (n)", "T. Secuencial (s)"]

display(HTML("<h4 style='color: #333;'>Resultados - Implementación Secuencial</h4>"))
display(df_1a.style.format({
    "Tamaño (n)": "{:,}",
    "T. Secuencial (s)": "{:.4f}"
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#4CAF50'), ('color', 'white'), ('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'center')]}
]).hide(axis="index"))

Sum(sqrt(x) + log(x + 1), (x, 1, n))

Tamaño (n),T. Secuencial (s)
"1,000,000",0.1168
"3,000,000",0.3613
"6,000,000",0.7335
"7,000,000",0.8623
"8,000,000",1.0210
"10,000,000",1.2250


### 1b) Versión vectorizada con NumPy

In [9]:
# -------------------------------------------------------
# Función 1b:
# -------------------------------------------------------
def suma_transformacion_vectorizada(n: int) -> float:
    """Calcula la misma suma usando operaciones vectoriales de NumPy."""
    valores = np.arange(1, n + 1, dtype=np.float64)
    return float(np.sqrt(valores).sum() + np.log1p(valores).sum())
    # np.log1p(x) = log(1+x) con mayor precisión numérica

tabla_1 = []
for fila in resultados_1a:
    n      = fila['n']
    t_sec  = fila['t_sec']
    t_vec  = medir_tiempo(suma_transformacion_vectorizada, n)
    speedup = t_sec / t_vec if t_vec > 0 else float('inf')
    tabla_1.append({'n': n, 't_sec': t_sec, 't_vec': t_vec, 'speedup': speedup})

# -------------------------------------------------------
# 3. Visualización
# -------------------------------------------------------
df_final = pd.DataFrame(tabla_1)
df_final.columns = ["Tamaño (n)", "T. Secuencial (s)", "T. Vectorizado (s)", "Speedup (S)"]

display(HTML("<h3 style='color: #2E86C1;'>Comparativa de Rendimiento Final:</h3>"))

# Estilo para la tabla final con colores y formato
estilo_final = df_final.style.format({
    "Tamaño (n)": "{:,}",
    "T. Secuencial (s)": "{:.4f}",
    "T. Vectorizado (s)": "{:.4f}",
    "Speedup (S)": "{:.2f}x"
}).background_gradient(
    # Agrega un mapa de calor para la columna Speedup (verde indica mejor aceleración)
    subset=["Speedup (S)"], cmap="Greens"
).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2874A6'), ('color', 'white'), ('font-size', '14px'), ('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'center'), ('font-size', '13px'), ('border', '1px solid #ddd')]}
]).hide(axis="index")

display(estilo_final)

Tamaño (n),T. Secuencial (s),T. Vectorizado (s),Speedup (S)
"1,000,000",0.1168,0.0139,8.43x
"3,000,000",0.3613,0.0391,9.23x
"6,000,000",0.7335,0.0773,9.49x
"7,000,000",0.8623,0.0891,9.68x
"8,000,000",1.0210,0.1037,9.85x
"10,000,000",1.2250,0.1301,9.42x


Conclusión ejercicio 1:

Como se puede ver, la vectorización con numpy agiliza bastante el proceso (teniendo en todos los casos al menos un speedup del 8x respecto al metodo tradicional de python), esto debido a que numpy utiliza instrucciones SIMD para procesar múltiples valores simultáneamente dentro de los registros vectoriales de un solo núcleo de la CPU, a diferencia del bucle tradicional de Python que procesa uno por uno los elementos y con un alto overhead del intérprete.

---
## Ejercicio 2 — Paralelismo práctico y limitaciones
**Propósito:** Explorar speedup y eficiencia reales con hilos (IO-bound) y procesos (CPU-bound).

### 2a) Paralelismo IO-bound con ThreadPoolExecutor

In [17]:
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from IPython.display import display, HTML



LATENCIA_SIMULADA = 0.04   # segundos (simula una llamada a API / lectura de archivo)
NUM_TAREAS        = 32     # número de solicitudes independientes a procesar

def tarea_io_bound(identificador: int) -> dict:
    """Simula una tarea con latencia de E/S seguida de un cómputo ligero."""
    time.sleep(LATENCIA_SIMULADA)                 # espera IO simulada
    resultado = sum(i * 0.1 for i in range(500))  # cómputo pequeño post-IO
    return {'id': identificador, 'valor': resultado}

def ejecutar_con_hilos(num_hilos: int, tareas: list) -> float:
    """Ejecuta todas las tareas con un pool de hilos y devuelve el tiempo total."""
    inicio = time.perf_counter()
    with ThreadPoolExecutor(max_workers=num_hilos) as executor:
        list(executor.map(tarea_io_bound, tareas))
    return time.perf_counter() - inicio

tareas = list(range(NUM_TAREAS))

# Ejecución secuencial (referencia)
t_inicio = time.perf_counter()
for t in tareas:
    tarea_io_bound(t)
t_secuencial_io = time.perf_counter() - t_inicio

# Mostrar resumen secuencial
display(HTML(f"<h4 style='color: #ddd;'>Tiempo secuencial (Referencia): <b>{t_secuencial_io:.3f} s</b> ({NUM_TAREAS} tareas × {LATENCIA_SIMULADA*1000:.0f} ms)</h4>"))

# Procesamiento en paralelo
tabla_2a = []
for p in [1, 2, 4, 8]:
    t_par  = ejecutar_con_hilos(p, tareas)
    S      = t_secuencial_io / t_par
    E      = S / p
    tabla_2a.append({'Hilos (p)': p, 'T. Paralelo (s)': t_par, 'Speedup (S)': S, 'Eficiencia (E)': E})

# -------------------------------------------------------
# Visualización
# -------------------------------------------------------
df_2a = pd.DataFrame(tabla_2a)
display(HTML("<h3 style='color: #2E86C1;'>Resultados de Paralelismo (IO-bound):</h3>"))

estilo_2a = df_2a.style.format({
    "T. Paralelo (s)": "{:.4f}",
    "Speedup (S)": "{:.2f}x",
    "Eficiencia (E)": "{:.2%}"
}).background_gradient(
    subset=["Speedup (S)"], cmap="Greens"
).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2874A6'), ('color', 'white'), ('font-size', '14px'), ('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'center'), ('font-size', '13px'), ('border', '1px solid #444')]}
]).hide(axis="index")

display(estilo_2a)

Hilos (p),T. Paralelo (s),Speedup (S),Eficiencia (E)
1,1.2971,1.00x,99.83%
2,0.6486,2.00x,99.82%
4,0.3250,3.98x,99.60%
8,0.1638,7.90x,98.80%


**Nota sobre el resultado 2a:** Para tareas IO-bound los hilos son efectivos porque mientras un hilo espera (sleep/IO), otros pueden ejecutarse. El GIL de Python no es un obstáculo aquí porque el hilo libera el GIL durante operaciones de IO.

### 2b) Paralelismo CPU-bound con multiprocessing

In [18]:
import time
import math
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from IPython.display import display, HTML
from herramientas import integrar_trapecios # Importación crucial para evitar problemas de ejecución.

# -------------------------------------------------------
# 1. Funciones matemáticas (Definidas en la misma celda)
# -------------------------------------------------------

def generar_segmentos(n_segmentos: int, N_por_segmento: int = 3_000_000) -> list:
    largo = 100.0 / n_segmentos
    return [(i * largo, (i + 1) * largo, N_por_segmento) for i in range(n_segmentos)]

# -------------------------------------------------------
# 2. Bloque Principal (Protegido con __main__)
# -------------------------------------------------------
if __name__ == '__main__':
    N_SEGMENTOS = 8
    segmentos   = generar_segmentos(N_SEGMENTOS)

    # --- Ejecución Secuencial ---
    t0 = time.perf_counter()
    for seg in segmentos:
        integrar_trapecios(seg)
    t_secuencial_cpu = time.perf_counter() - t0

    # Mostrar resumen secuencial
    display(HTML(f"<h4 style='color: #ddd;'>Tiempo secuencial (Referencia): <b>{t_secuencial_cpu:.3f} s</b> ({N_SEGMENTOS} segmentos)</h4>"))

    # --- Procesamiento en Paralelo (ProcessPoolExecutor) ---
    tabla_2b = []

    # ¡Aquí agregamos el 8 a la lista de procesos!
    for p in [1, 2, 4, 8]:
        t0 = time.perf_counter()

        # Multiprocesamiento real
        with ProcessPoolExecutor(max_workers=p) as executor:
            list(executor.map(integrar_trapecios, segmentos))

        t_par = time.perf_counter() - t0
        S     = t_secuencial_cpu / t_par
        E     = S / p

        tabla_2b.append({'Procesos (p)': p, 'T. Paralelo (s)': t_par, 'Speedup (S)': S, 'Eficiencia (E)': E})

    # -------------------------------------------------------
    # 3. Visualización Vistosa de Resultados
    # -------------------------------------------------------
    df_2b = pd.DataFrame(tabla_2b)
    display(HTML("<h3 style='color: #2E86C1;'>Resultados con Multiprocesamiento (CPU-bound real):</h3>"))

    estilo_2b = df_2b.style.format({
        "T. Paralelo (s)": "{:.4f}",
        "Speedup (S)": "{:.2f}x",
        "Eficiencia (E)": "{:.2%}"
    }).background_gradient(
        subset=["Speedup (S)"], cmap="Greens"
    ).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2874A6'), ('color', 'white'), ('font-size', '14px'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center'), ('font-size', '13px'), ('border', '1px solid #444')]}
    ]).hide(axis="index")

    display(estilo_2b)

Procesos (p),T. Paralelo (s),Speedup (S),Eficiencia (E)
1,3.8277,0.96x,96.30%
2,2.0184,1.83x,91.31%
4,1.1157,3.30x,82.60%
8,0.8607,4.28x,53.54%


### Conclusión ejercicio 2:

Este ejercicio concuerda con la Ley de Amdahl, la cual establece que el speedup máximo de un programa está estrictamente limitado por su fracción secuencial. Tal como muestran los resultados, se puede ver un aumento positivo y constante en el speedup, llegando a un 4.28x con 8 procesos. Aunque el crecimiento no es lineal debido a la fracción secuencial forzada por el overhead del sistema operativo. Este costo administrativo es el que impide alcanzar un speedup ideal de 8x.

También cumple con la ley de Gustafson, la cual establece que
la eficiencia solo se puede mantener alta si el tamaño del problema crece a medida que aumentamos los procesos. 
Los resultados muestran que al mantener una carga de trabajo fija, la Eficiencia experimenta un descenso natural, pasando del 96.30% (con 1 proceso) al 53.54% (con 8 procesos). Esto ocurre porque el costo de gestionar múltiples procesos empieza a ocupar una proporción mayor del tiempo total en comparación con el cálculo matemático puro.

---
## Ejercicio 3 — Decisión estratégica
**Propósito:** Aplicar las métricas a un caso realista de ciencia de datos (procesamiento de lotes independientes) y formular una recomendación técnica fundamentada.

### 3a) Procesamiento por lotes independiente

In [15]:
import time
import math
import random
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from IPython.display import display, HTML
from procesamiento import procesar_lote_datos # Importación crucial para evitar problemas de ejecución.

# -------------------------------------------------------
# Bloque Principal
# -------------------------------------------------------
if __name__ == '__main__':
    # Aumentamos el tamaño a 1,200,000 para "hacer sudar" a la CPU y que valga la pena paralelizar
    lotes = [{'semilla': i, 'tamanio': 1_200_000} for i in range(8)]

    # --- Secuencial ---
    t0    = time.perf_counter()
    _     = [procesar_lote_datos(l) for l in lotes]
    t_sec = time.perf_counter() - t0

    display(HTML(f"<h4 style='color: #ddd;'>Tiempo secuencial (Referencia): <b>{t_sec:.3f} s</b> ({len(lotes)} lotes)</h4>"))

    # --- Paralelo (ProcessPoolExecutor) ---
    tabla_3 = []
    for p in [1, 2, 4]:
        t0    = time.perf_counter()
        with ProcessPoolExecutor(max_workers=p) as executor:
            list(executor.map(procesar_lote_datos, lotes))
        t_par = time.perf_counter() - t0
        S     = t_sec / t_par
        E     = S / p
        tabla_3.append({'Procesos (p)': p, 'T. Paralelo (s)': t_par, 'Speedup (S)': S, 'Eficiencia (E)': E})

    # --- Visualización ---
    df_3 = pd.DataFrame(tabla_3)
    display(HTML("<h3 style='color: #2E86C1;'>Resultados de Procesamiento por Lotes:</h3>"))

    estilo_3 = df_3.style.format({
        "T. Paralelo (s)": "{:.4f}",
        "Speedup (S)": "{:.2f}x",
        "Eficiencia (E)": "{:.2%}"
    }).background_gradient(
        subset=["Speedup (S)"], cmap="Greens"
    ).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2874A6'), ('color', 'white'), ('font-size', '14px'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center'), ('font-size', '13px'), ('border', '1px solid #444')]}
    ]).hide(axis="index")

    display(estilo_3)

Procesos (p),T. Paralelo (s),Speedup (S),Eficiencia (E)
1,7.0674,0.97x,97.31%
2,3.9284,1.75x,87.53%
4,2.5236,2.73x,68.13%


### 3b) Diagnóstico y recomendación técnica

In [16]:
# -------------------------------------------------------
# Resumen automático de métricas y Recomendación Visual
# -------------------------------------------------------
mejor_config = max(tabla_3, key=lambda x: x['Speedup (S)'])
p_opt = mejor_config['Procesos (p)']
S_opt = mejor_config['Speedup (S)']
E_opt = mejor_config['Eficiencia (E)']

# Lógica de decisión
if S_opt >= 1.8 and E_opt >= 0.6:
    recomendacion = 'PARALELIZAR'
    color = '#27AE60' # Verde
    desc = 'El speedup y la eficiencia justifican plenamente el overhead operativo.'
elif S_opt >= 1.3:
    recomendacion = 'PARALELO MODERADO'
    color = '#F39C12' # Naranja
    desc = 'Existe beneficio, pero el overhead de comunicación es alto. Evaluar tamaño de lotes.'
else:
    recomendacion = 'SECUENCIAL'
    color = '#E74C3C' # Rojo
    desc = 'El overhead supera la ganancia matemática. Mantener en un solo núcleo.'

# Renderizado de una tarjeta HTML
html_resumen = f"""
<div style="border: 2px solid {color}; border-radius: 8px; padding: 15px; background-color: #222; color: #eee; width: 80%;">
    <h3 style="color: {color}; margin-top: 0;">Resumen de Métricas - Ejercicio 3</h3>
    <ul style="font-size: 14px;">
        <li><b>Tiempo secuencial:</b> {t_sec:.3f} s</li>
        <li><b>Mejor configuración:</b> p = {p_opt} procesos</li>
        <li><b>Speedup Óptimo (S):</b> {S_opt:.2f}x</li>
        <li><b>Eficiencia (E):</b> {E_opt:.2%}</li>
    </ul>
    <hr style="border-color: #444;">
    <h4 style="color: {color}; margin-bottom: 5px;">Recomendación: {recomendacion}</h4>
    <p style="margin-top: 0; font-size: 13px;"><i>{desc}</i></p>
</div>
"""

display(HTML(html_resumen))

### Recomendación Ejercicio 3:
Al obtener un Speedup de 2.73x y una Eficiencia (E) del 68.13% con 4 procesos, la recomendación es paralelizar. Los resultados demuestran que la ganancia en tiempo justifica el costo operativo de gestionar múltiples núcleos. Esta mejora se fundamenta en tres factores: Al distribuir la carga entre 4 procesos se logra un tiempo decente para realizar tareas, produce una eficiencia del 68.13% indicando que el volumen de datos es suficiente para amortizar el costo de comunicación entre procesos y el aumento en la carga computacional por lote permite que la CPU dedique más tiempo al cálculo real que a la gestión administrativa.


## Conclusión final


La optimización del rendimiento en ciencia de datos no es una solución única, sino una decisión estratégica basada en métricas de escalabilidad y en el entendimiento profundo de las limitaciones del hardware. A través de este laboratorio, hemos validado que la eficiencia depende de la naturaleza de la carga de trabajo y del "peaje" de coordinación del sistema operativo, estableciendo como prioridad técnica la regla de "vectorizar primero, paralelizar después". La vectorización con NumPy demostró ser la optimización más eficiente al aprovechar instrucciones SIMD y localidad de caché en un solo núcleo, logrando aceleraciones superiores a 9x sin el costo administrativo de gestionar procesos. El ejercicio con multiprocesamiento confirmó que el Speedup (S) está estrictamente limitado por la fracción secuencial forzada por el overhead de pickling y coordinación, tal como dicta la Ley de Amdahl. No obstante, bajo la Ley de Gustafson, comprobamos que al escalar el tamaño del problema hasta 1,200,000 registros, el paralelismo se vuelve rentable, logrando una eficiencia del 68.13% con 4 procesos que justifica plenamente la distribución de la carga. En conclusión, la recomendación de paralelizar debe ser estrictamente cuantitativa y solo se justifica cuando el tiempo de cómputo real eclipsa la latencia de comunicación entre procesos, evitando el desperdicio de recursos por excesivos cambios de contexto (Context Switching). Un arquitecto de sistemas eficiente debe medir primero, agotar las optimizaciones algorítmicas y recurrir al paralelismo horizontal solo cuando el volumen y la independencia de los datos garantizan un retorno de inversión real en el tiempo de ejecución.

